# Undertow: quantum repair planning for roads failing from below

**What this notebook shows**
1. The repair-planning problem written as a QUBO, then as a qubit Hamiltonian
2. The QAOA circuit, built explicitly with `QAOAAnsatz`
3. Transpilation for a real IBM device (FakeTorino, 133-qubit Heron), and what it costs in depth and two-qubit gates
4. Parameter training with the `SamplerV2` primitive and a CVaR objective
5. Sampling with `SamplerV2`: ideal simulator vs. noisy simulator (real-device noise model) vs. noisy + error suppression
6. Comparison against three baselines: random guessing, the exact classical optimum, and today's policy (fix the roads with the most 311 complaints)

Run top to bottom. Takes about 2 minutes on a laptop. Charts and tables are saved to `results/`.

In [1]:
import sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler import generate_preset_pass_manager, PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling
from qiskit.circuit.library import XGate
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2, SamplerV2
from qiskit_optimization.translators import to_ising
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_optimization.converters import QuadraticProgramToQubo

from undertow.data_synthetic import make_grid_network, adjacency_pairs
from undertow.risk import add_scores
from undertow.qubo import build_program, build_lean_program, tune_price, plan_value
from undertow.solve import solve_exact, solve_current_policy
from undertow.metrics import score_plan
from undertow.charts import evac_route_chart, good_plans_chart, training_chart

# Chart style: thin marks, recessive grid, text in neutral ink
BLUE, ORANGE, AQUA, GRAY, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#9a9893", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#d6d5d0",
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": "#ebeae6", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11, "axes.titlesize": 13,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
})
SEED = 7

## 1. The data

Each row is a road segment. `risk` is the hidden-damage score (tide exposure, groundwater depth, elevation),
`criticality` is higher on the evacuation route and hospital access, and `value = risk x criticality` is the harm we avoid by repairing it.

If Dev A has produced `data/segments.csv`, it's used automatically. Otherwise we use the synthetic Miami Beach-style network.

**Size knobs:** `K` segments go to the quantum optimizer. In the lean formulation, qubits = `K`.
Lean formulation: qubits = K. K=8 runs everything in about a minute; try 10-12.

In [2]:
K, BUDGET, REPS = 8, 6, 3

csv = ROOT / "data" / "segments.csv"
df = pd.read_csv(csv) if csv.exists() else make_grid_network()[1]
print("data:", csv.name if csv.exists() else "synthetic network")
df = add_scores(df)
pairs_all = adjacency_pairs(df)

# Hybrid step: the classical risk model shortlists the K most valuable candidates.
cand = df.sort_values("value", ascending=False).head(K)
ids = set(cand.segment_id)
pairs = [(a, b) for a, b in pairs_all if a in ids and b in ids]
print(f"{len(df)} segments in the area, {K} candidates sent to the optimizer, budget = {BUDGET} cost units")
cand[["segment_id", "risk", "criticality", "value", "repair_cost", "reports_311", "evac_route", "hospital_access"]]

data: synthetic network
17 segments in the area, 8 candidates sent to the optimizer, budget = 6 cost units


,segment_id,risk,criticality,value,repair_cost,reports_311,evac_route,hospital_access
7,S07,0.857,2.5,2.142,1,6,True,False
9,S09,0.464,2.5,1.160,3,3,True,False
8,S08,0.909,1.0,0.909,3,4,False,False
1,S01,0.902,1.0,0.902,3,3,False,False
0,S00,0.846,1.0,0.846,2,2,False,False
14,S14,0.780,1.0,0.780,3,1,False,False
10,S10,0.745,1.0,0.745,3,2,False,False
2,S02,0.697,1.0,0.697,2,3,False,False


**What this means:** each row is a road segment. Higher `risk` means more hidden damage under the surface, and `value` combines that risk with how important the road is for evacuation or hospital access.

## 2. From repair plan to qubits

Binary variable `x_i = 1` means repair segment `i`.

$$\max \sum_i v_i x_i + \sum_{(i,j)\ \text{adjacent}} s_{ij} x_i x_j \quad \text{s.t.} \quad \sum_i c_i x_i \le B$$

The pairwise term `s_ij` is why this fits quantum optimization naturally: repairing two connected blocks of the
same evacuation route is worth more than the two repairs separately.

We tried two encodings of the budget:
- **Slack (first attempt):** `QuadraticProgramToQubo` adds slack qubits and a big penalty. Exact, but it costs extra qubits
  and makes the energy landscape hard for shallow QAOA.
- **Lean (what we use):** one qubit per segment. The budget becomes a **price per cost unit** λ
  (tuned by bisection: the lowest price whose best plan fits the budget), and over-budget plans are filtered out after sampling.

In [3]:
qp = build_program(cand, pairs, BUDGET)             # the true constrained problem (for the exact baseline)
price = tune_price(cand, pairs, BUDGET)
lean = build_lean_program(cand, pairs, price)
cost_op, offset = to_ising(lean)
n = cost_op.num_qubits
slack_conv = QuadraticProgramToQubo()
slack_op, slack_offset = slack_conv.convert(qp).to_ising()
print(qp.export_as_lp_string())
print(f"price per cost unit λ = {price:.3f}")
print(f"lean: {n} qubits, {len(cost_op)} Pauli terms | slack: {slack_op.num_qubits} qubits, {len(slack_op)} Pauli terms")

\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: undertow_repair_plan

Maximize
 obj: 2.142000000000 S07 + 1.160000000000 S09 + 0.909000000000 S08
      + 0.902000000000 S01 + 0.846000000000 S00 + 0.780000000000 S14
      + 0.745000000000 S10 + 0.697000000000 S02 + [ 0.928000000000 S07*S09
      + 0.428000000000 S07*S08 + 0.428000000000 S07*S01 + 0.372000000000 S07*S10
      + 0.232000000000 S09*S10 + 0.452000000000 S08*S01 + 0.390000000000 S08*S14
      + 0.424000000000 S01*S00 + 0.348000000000 S00*S02 + 0.372000000000 S14*S10
      ]/2
Subject To
 budget: S07 + 3 S09 + 3 S08 + 3 S01 + 2 S00 + 3 S14 + 3 S10 + 2 S02 <= 6

Bounds
 0 <= S07 <= 1
 0 <= S09 <= 1
 0 <= S08 <= 1
 0 <= S01 <= 1
 0 <= S00 <= 1
 0 <= S14 <= 1
 0 <= S10 <= 1
 0 <= S02 <= 1

Binaries
 S07 S09 S08 S01 S00 S14 S10 S02
End

price per cost unit λ = 0.440
lean: 8 qubits, 18 Pauli terms | slack: 11 qubits, 66 Pauli terms


## 3. Baselines (the story needs these)

- **Exact classical optimum**: brute-force eigensolver. At this size it is instant and always right.
- **Current policy**: repair the segments with the most 311 pothole complaints until the budget runs out.
- **Random guessing**: the chance a uniformly random bitstring is the optimal plan. QAOA has to beat this to be doing anything.

In [4]:
exact = solve_exact(qp)
policy = solve_current_policy(df, BUDGET)
best_value = plan_value(cand, pairs, exact.chosen)

costs = cand.set_index("segment_id")["repair_cost"].to_dict()
names = [v.name for v in lean.variables]

def decode(bitstring, var_names=names):
    # Qiskit bitstrings are little-endian: the rightmost bit is qubit 0
    chosen = {nm for nm, b in zip(var_names, bitstring[::-1]) if b == "1" and nm in costs}  # drop slack qubits
    return chosen, sum(costs[s] for s in chosen) <= BUDGET

def evaluate(counts, var_names=names):
    total = sum(counts.values())
    feasible = optimal = good = 0
    best = 0.0
    for bs, c in counts.items():
        chosen, ok = decode(bs, var_names)
        if ok:
            v = plan_value(cand, pairs, chosen)
            feasible += c
            best = max(best, v)
            optimal += c * (chosen == exact.chosen)
            good += c * (v >= 0.9 * best_value)
    return {"P(feasible plan)": feasible / total, "P(optimal plan)": optimal / total,
            "P(good plan, >=90% of optimum)": good / total,
            "best plan found": round(best, 3), "best / optimum": round(best / best_value, 3)}

# Random guessing: every bitstring equally likely
uniform = {format(i, f"0{n}b"): 1 for i in range(2 ** n)}   # every possible plan equally likely
random_row = evaluate(uniform)
print("Optimal plan:", sorted(exact.chosen), "value", best_value)
print("Current policy picks:", sorted(policy.chosen))
print(f"A random plan is optimal {random_row['P(optimal plan)']:.2%} of the time and good {random_row['P(good plan, >=90% of optimum)']:.2%} of the time")

Optimal plan: ['S00', 'S07', 'S09'] value 4.612
Current policy picks: ['S03', 'S07', 'S12']
A random plan is optimal 0.39% of the time and good 1.17% of the time


**What this means:** this is the yardstick. The exact classical answer is the best possible plan. Random guessing and today's approach are what the optimizer has to beat.

## 4. The QAOA circuit

`QAOAAnsatz` alternates a **cost layer** (rotations from our Hamiltonian, `exp(-i γ H_C)`) and a **mixer layer**
(`exp(-i β Σ X)`), repeated `REPS` times. The optimizer tunes the `γ` and `β` angles.
The full circuit is too wide to read, so the diagram below uses a 3-segment toy version of the same problem.

In [5]:
ansatz = QAOAAnsatz(cost_op, reps=REPS)
print(f"QAOA ansatz: {ansatz.num_qubits} qubits, {ansatz.num_parameters} parameters (γ, β per layer)")

# Readable toy circuit for the slides
toy = cand.head(3)
toy_ids = set(toy.segment_id)
toy_op, _ = to_ising(build_lean_program(toy, [(a, b) for a, b in pairs if a in toy_ids and b in toy_ids], price))
toy_circ = QAOAAnsatz(toy_op, reps=1).decompose(reps=2)
fig = toy_circ.draw("mpl", fold=-1, style="iqp")
fig.savefig(RESULTS / "qaoa_circuit_toy.png", dpi=160, bbox_inches="tight")
fig

QAOA ansatz: 8 qubits, 6 parameters (γ, β per layer)


<Figure size 1123.61x284.278 with 1 Axes>

## 5. Transpilation for a real IBM device

Real hardware only has certain native gates (for Heron: `cz`, `rz`, `sx`, `x`) and limited qubit connectivity.
The transpiler rewrites the circuit to fit, which adds SWAPs and depth. More depth = more noise.
We compare optimization levels 0-3 on FakeTorino, a snapshot of a real 133-qubit IBM device.

In [6]:
backend = FakeTorino()
measured = ansatz.measure_all(inplace=False)

rows = []
isa = {}
for level in range(4):
    pm = generate_preset_pass_manager(optimization_level=level, backend=backend, seed_transpiler=SEED)
    t = time.perf_counter()
    circ = pm.run(measured)
    ops = circ.count_ops()
    isa[level] = circ
    rows.append({"optimization level": level, "depth": circ.depth(),
                 "two-qubit gates": ops.get("cz", 0) + ops.get("ecr", 0) + ops.get("cx", 0),
                 "transpile seconds": round(time.perf_counter() - t, 2)})
transpile_table = pd.DataFrame(rows).set_index("optimization level")
transpile_table.to_csv(RESULTS / "transpilation.csv")
print(f"Before transpiling: depth {measured.decompose(reps=3).depth()} in abstract gates")
transpile_table

Before transpiling: depth 56 in abstract gates


,depth,two-qubit gates,transpile seconds
optimization level,,,
0,354,165,0.01
1,317,105,0.01
2,257,98,0.01
3,260,100,0.01


**What this means:** real quantum computers can only run a few basic operations, so our circuit gets rewritten. Fewer two-qubit operations means less noise and a more reliable answer.

## 6. Train the angles with `EstimatorV2`

The classical optimizer (COBYLA) proposes angles; the Estimator primitive returns the expected energy
`⟨ψ(γ,β)|H_C|ψ(γ,β)⟩`. Lower energy = better plans. We train on the ideal simulator with 3 random starts per formulation,
then sample each with `SamplerV2`. This is also where we compare the slack and lean encodings.
On real hardware this loop would run inside a Qiskit Runtime session.

In [7]:
sim = AerSimulator(seed_simulator=SEED)
sim_pm = generate_preset_pass_manager(optimization_level=1, backend=sim)
estimator = EstimatorV2(options={"run_options": {"seed": SEED}})
ideal_sampler = SamplerV2(seed=SEED)

def train(op, reps, starts=3, maxiter=200):
    circ = sim_pm.run(QAOAAnsatz(op, reps=reps))
    histories, best = [], None
    for start in range(starts):
        hist = []
        def energy(theta):
            e = float(estimator.run([(circ, op, theta)]).result()[0].data.evs)
            hist.append(e)
            return e
        x0 = np.random.default_rng(start).uniform(0, np.pi, circ.num_parameters)
        res = minimize(energy, x0, method="COBYLA", options={"maxiter": maxiter})
        histories.append(hist)
        if best is None or res.fun < best.fun:
            best = res
    return best.x, histories

t = time.perf_counter()
theta_opt, histories = train(cost_op, REPS)
theta_slack, _ = train(slack_op, REPS)
print(f"trained both formulations in {time.perf_counter() - t:.1f}s")

# Slack vs lean on the ideal simulator
slack_names = [v.name for v in slack_conv.convert(qp).variables]
def sample_ideal(op, theta, reps):
    c = sim_pm.run(QAOAAnsatz(op, reps=reps).measure_all(inplace=False)).assign_parameters(theta)
    return ideal_sampler.run([c], shots=4000).result()[0].data.meas.get_counts()
cmp_rows = []
for label, op, th, vn in [("Slack (first attempt)", slack_op, theta_slack, slack_names), ("Lean (one qubit per segment)", cost_op, theta_opt, names)]:
    l3 = generate_preset_pass_manager(optimization_level=3, backend=FakeTorino(), seed_transpiler=SEED).run(
        QAOAAnsatz(op, reps=REPS).measure_all(inplace=False))
    ev = evaluate(sample_ideal(op, th, REPS), vn)
    cmp_rows.append({"formulation": label, "qubits": op.num_qubits,
                     "two-qubit gates on device": l3.count_ops().get("cz", 0),
                     "P(good plan), ideal": ev["P(good plan, >=90% of optimum)"],
                     "x better than random": round(ev["P(good plan, >=90% of optimum)"] / random_row["P(good plan, >=90% of optimum)"], 1)})
formulations = pd.DataFrame(cmp_rows).set_index("formulation")
formulations.to_csv(RESULTS / "formulation_comparison.csv")

fig = training_chart(histories)
fig.savefig(RESULTS / "qaoa_training.png", dpi=160, bbox_inches="tight")
formulations.style.format({"P(good plan), ideal": "{:.1%}"})

trained both formulations in 8.7s


,qubits,two-qubit gates on device,"P(good plan), ideal",x better than random
formulation,,,,
Slack (first attempt),11,578,2.3%,2.000000
Lean (one qubit per segment),8,100,18.4%,15.700000


**What this means:** the first table compares two ways of writing the problem for a quantum computer. The simpler one needs fewer qubits and fewer operations, so it works much better. The chart shows the optimizer getting better at picking repairs with each step.

## 7. Sample with `SamplerV2`: ideal vs. noisy vs. error suppression

Same trained circuit, transpiled for FakeTorino, run four ways (4,000 shots each):

| Run | What it models |
|---|---|
| Ideal | A perfect quantum computer (noiseless simulation of the device-ready circuit) |
| Noisy, level 1 | The real device's noise (gate errors, readout errors, T1/T2) with light compilation |
| Noisy, level 3 | Same noise, heaviest compilation: fewer gates = less noise (**error suppression by compilation**) |
| Noisy, level 3 + DD | Adds **dynamical decoupling**: X-X pulses on idle qubits to cancel slow dephasing |

We also apply a simple **post-selection** step to every run: bitstrings that break the budget are thrown out
(that's what "best plan found" uses).

In [8]:
noisy_sampler = SamplerV2.from_backend(backend, seed=SEED)
ideal_sampler = SamplerV2(seed=SEED)

dd_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=SEED)
dd_pm.scheduling = PassManager([
    ALAPScheduleAnalysis(target=backend.target),
    PadDynamicalDecoupling(target=backend.target, dd_sequence=[XGate(), XGate()]),
])

runs = {
    "Ideal simulator": (ideal_sampler, isa[1]),
    "Noisy, level 1": (noisy_sampler, isa[1]),
    "Noisy, level 3": (noisy_sampler, isa[3]),
    "Noisy, level 3 + DD": (noisy_sampler, dd_pm.run(measured)),
}
rows, counts_by_run = [], {}
for name, (sampler, circ) in runs.items():
    t = time.perf_counter()
    counts = sampler.run([circ.assign_parameters(theta_opt)], shots=4000).result()[0].data.meas.get_counts()
    counts_by_run[name] = counts
    rows.append({"run": name, **evaluate(counts), "seconds": round(time.perf_counter() - t, 1)})
rows.append({"run": "Random guessing", **random_row, "seconds": 0})
results = pd.DataFrame(rows).set_index("run")
results["x better than random"] = (results["P(good plan, >=90% of optimum)"] / random_row["P(good plan, >=90% of optimum)"]).round(1)
results.to_csv(RESULTS / "sampling_results.csv")
results.style.format({"P(feasible plan)": "{:.1%}", "P(optimal plan)": "{:.2%}", "P(good plan, >=90% of optimum)": "{:.1%}"})

,P(feasible plan),P(optimal plan),"P(good plan, >=90% of optimum)",best plan found,best / optimum,seconds,x better than random
run,,,,,,,
Ideal simulator,35.8%,16.07%,18.6%,4.612000,1.000000,0.100000,15.900000
"Noisy, level 1",31.9%,9.18%,11.8%,4.612000,1.000000,1.700000,10.100000
"Noisy, level 3",33.1%,9.93%,12.7%,4.612000,1.000000,1.000000,10.800000
"Noisy, level 3 + DD",33.3%,8.70%,11.8%,4.612000,1.000000,2.500000,10.000000
Random guessing,18.8%,0.39%,1.2%,4.612000,1.000000,0.000000,1.000000


**What this means:** each row is a different way of running the same optimizer. The higher the "x better than random" number, the more often it finds a good repair plan.

In [9]:
p_good = (results["P(good plan, >=90% of optimum)"] * 100).to_dict()
fig = good_plans_chart(p_good)
fig.savefig(RESULTS / "p_good_plan.png", dpi=160, bbox_inches="tight")
good_plans_chart(p_good, slide=True).savefig(RESULTS / "slide_good_plans_vs_random.png", dpi=120)  # 1920x1080 for slides
fig


<Figure size 950x440 with 1 Axes>

**What this means:** a quantum optimizer tried at random would almost never land on a good repair plan. Here it does so many times more often, even with the noise a real quantum computer has.

## 8. The answer that matters: the repair plan

Every plan is scored the same way on the **whole** area (not just the shortlist).
`evac_route_open_%` is the chance every block of the evacuation route survives the next king tide.

In [10]:
def best_sampled_plan(counts):
    best, plan = -1, set()
    for bs in counts:
        chosen, ok = decode(bs)
        if ok and (v := plan_value(cand, pairs, chosen)) > best:
            best, plan = v, chosen
    return plan

plans = {
    "Current policy (most 311 reports)": policy.chosen,
    "Classical exact": exact.chosen,
    "QAOA, ideal simulator": best_sampled_plan(counts_by_run["Ideal simulator"]),
    "QAOA, noisy (level 3)": best_sampled_plan(counts_by_run["Noisy, level 3"]),
}
plan_table = pd.DataFrame({k: score_plan(df, pairs_all, v) for k, v in plans.items()}).T
plan_table["repairs"] = [" ".join(sorted(v)) for v in plans.values()]
plan_table.to_csv(RESULTS / "plan_comparison.csv")

evac = plan_table["evac_route_open_%"].astype(float).to_dict()
fig = evac_route_chart(evac)
fig.savefig(RESULTS / "evac_route_open.png", dpi=160, bbox_inches="tight")
evac_route_chart(evac, slide=True).savefig(RESULTS / "slide_evac_route_open.png", dpi=120)  # 1920x1080 for slides
plan_table


,segments_repaired,cost_used,objective,hidden_risk_removed,expected_failures_left,evac_route_open_%,hospital_access_open_%,repairs
Current policy (most 311 reports),3.0,5.0,3.240,1.80,7.22,44.7,83.4,S03 S07 S12
Classical exact,3.0,6.0,4.612,2.17,6.85,79.1,83.4,S00 S07 S09
"QAOA, ideal simulator",3.0,6.0,4.612,2.17,6.85,79.1,83.4,S00 S07 S09
"QAOA, noisy (level 3)",3.0,6.0,4.612,2.17,6.85,79.1,83.4,S00 S07 S09


**What this means:** with the same budget, the plan the optimizer picks protects far more of the evacuation route than fixing whatever gets the most complaints. The exact classical solver finds the same plan, which shows the quantum result is correct.

## 9. What the numbers say (auto-generated from this run)

In [11]:
r = results
ideal_x, noisy1_x, noisy3_x, dd_x = (r.loc[k, "x better than random"] for k in
    ["Ideal simulator", "Noisy, level 1", "Noisy, level 3", "Noisy, level 3 + DD"])
pol, qa = plan_table.loc["Current policy (most 311 reports)"], plan_table.loc["QAOA, ideal simulator"]
lines = [
    f"- On an ideal simulator, {r.loc['Ideal simulator', 'P(good plan, >=90% of optimum)']:.1%} of QAOA shots are good repair plans, {ideal_x}x the rate of random guessing.",
    f"- Real-device noise cuts that to {noisy1_x}x. Heavier compilation (level 3, {transpile_table.loc[3, 'two-qubit gates']} vs "
    f"{transpile_table.loc[1, 'two-qubit gates']} two-qubit gates) gives {noisy3_x}x; adding dynamical decoupling gives {dd_x}x.",
    f"- With post-selection, the best plan found reaches {r.loc['Noisy, level 3', 'best / optimum']:.0%} of the optimum under noise "
    f"({r.loc['Ideal simulator', 'best / optimum']:.0%} ideal).",
    f"- Versus today's policy, the optimized plan keeps the evacuation route open {qa['evac_route_open_%']:.0f}% of the time "
    f"instead of {pol['evac_route_open_%']:.0f}%, for the same budget.",
    f"- Honest caveat: at {n} qubits there are only {2**n:,} possible bitstrings, so 4,000 shots nearly enumerate them and a classical "
    "solver is instant. The point is that the formulation works and QAOA concentrates probability on good plans; quantum advantage "
    "would need far larger instances and lower noise.",
]
summary = "\n".join(lines)
(RESULTS / "summary.md").write_text(summary)
print(summary)

- On an ideal simulator, 18.6% of QAOA shots are good repair plans, 15.9x the rate of random guessing.
- Real-device noise cuts that to 10.1x. Heavier compilation (level 3, 100 vs 105 two-qubit gates) gives 10.8x; adding dynamical decoupling gives 10.0x.
- With post-selection, the best plan found reaches 100% of the optimum under noise (100% ideal).
- Versus today's policy, the optimized plan keeps the evacuation route open 79% of the time instead of 45%, for the same budget.
- Honest caveat: at 8 qubits there are only 256 possible bitstrings, so 4,000 shots nearly enumerate them and a classical solver is instant. The point is that the formulation works and QAOA concentrates probability on good plans; quantum advantage would need far larger instances and lower noise.


**Reading the dynamical decoupling result:** DD protects idle qubits from slow, coherent dephasing (for example crosstalk between
neighboring qubits). The simulator's noise model mostly contains gate errors, readout errors and T1/T2 relaxation, which DD can't
undo, so little or no improvement in simulation is expected. On real hardware DD often helps; that's a next step to test.

## 10. The size gap (for the canvas)

| | Segments | Qubits |
|---|---|---|
| This demo | 8-12 | 8-12 |
| A real neighborhood | fill in from `data/segments.csv` | = number of segments |
| A whole county | fill in (number of road segments in Miami-Dade) | thousands |

What would have to be true: deeper circuits with much lower error rates (or error correction), better ways to encode budgets
without slack qubits, real subgrade sensor or inspection data to validate the risk model, and a public-works department willing
to plan repairs by corridor instead of by complaint.

## Optional: run on real IBM hardware

If you have IBM Quantum access, swap the backend and sampler. Keep it to one small run; jobs queue.

```python
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler
service = QiskitRuntimeService()
hw = service.least_busy(operational=True, simulator=False)
circ = generate_preset_pass_manager(optimization_level=3, backend=hw).run(measured).assign_parameters(theta_opt)
job = RuntimeSampler(mode=hw).run([circ], shots=4000)
print(job.job_id()); hw_counts = job.result()[0].data.meas.get_counts()
evaluate(hw_counts)
```